In [43]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import re
import string
!pip install nltk
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [44]:
data = pd.read_csv('/kaggle/input/datasets/shashwatwork/consume-complaints-dataset-fo-nlp/complaints_processed.csv')
data

,Unnamed: 0,product,narrative
0,0,credit_card,purchase order day shipping amount receive pro...
1,1,credit_card,forwarded message date tue subject please inve...
2,2,retail_banking,forwarded message cc sent friday pdt subject f...
3,3,credit_reporting,payment history missing credit report speciali...
4,4,credit_reporting,payment history missing credit report made mis...
...,...,...,...
162416,162416,debt_collection,name
162417,162417,credit_card,name
162418,162418,debt_collection,name
162419,162419,credit_card,name


In [45]:
data.drop(columns='Unnamed: 0', inplace=True)

In [4]:
data.head(10)

,product,narrative
0,credit_card,purchase order day shipping amount receive pro...
1,credit_card,forwarded message date tue subject please inve...
2,retail_banking,forwarded message cc sent friday pdt subject f...
3,credit_reporting,payment history missing credit report speciali...
4,credit_reporting,payment history missing credit report made mis...
5,credit_reporting,payment history missing credit report made mis...
6,credit_reporting,va date complaint experian credit bureau invol...
7,credit_reporting,account reported abbreviated name full name se...
8,credit_reporting,account reported abbreviated name full name se...
9,credit_reporting,usdoexxxx account reported abbreviated name fu...


In [6]:
data['product'].value_counts()

product
credit_reporting       91179
debt_collection        23150
mortgages_and_loans    18990
credit_card            15566
retail_banking         13536
Name: count, dtype: int64

In [7]:
data.isna().sum()

product       0
narrative    10
dtype: int64

In [8]:
data[data['narrative']=='name']

,product,narrative
162415,debt_collection,name
162416,debt_collection,name
162417,credit_card,name
162418,debt_collection,name
162419,credit_card,name
162420,credit_reporting,name


In [46]:
data.dropna(subset=['narrative'], inplace=True)
data.drop(data[data['narrative'] == 'name'].index, inplace=True)
print(data[data['narrative']=='name'])
print(data.isna().sum())

Empty DataFrame
Columns: [product, narrative]
Index: []
product      0
narrative    0
dtype: int64


In [47]:
label_encoder = LabelEncoder()
data['label'] = label_encoder.fit_transform(data['product'])
data

,product,narrative,label
0,credit_card,purchase order day shipping amount receive pro...,0
1,credit_card,forwarded message date tue subject please inve...,0
2,retail_banking,forwarded message cc sent friday pdt subject f...,4
3,credit_reporting,payment history missing credit report speciali...,1
4,credit_reporting,payment history missing credit report made mis...,1
...,...,...,...
162410,credit_reporting,zales comenity bank closed sold account report...,1
162411,retail_banking,zelle suspended account without cause banking ...,4
162412,debt_collection,zero contact made debt supposedly resolved fou...,2
162413,mortgages_and_loans,zillow home loan nmls nmls actual quote provid...,3


In [48]:
def cleaning_data(text):
    text = str(text).lower()

    # إزالة الإيميلات واللينكات
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'.pic\S+', '', text)

    # الاحتفاظ بالحروف الإنجليزية فقط
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # إزالة علامات الترقيم
    text = "".join([i for i in text if i not in string.punctuation])

    # Tokenization
    words = word_tokenize(text)

    # إزالة Stop Words والكلمات القصيرة
    words = [word for word in words if word not in stop_words and len(word) > 2]

    text = " ".join(words)

    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [49]:
data['clean_narrative'] = data['narrative'].apply(cleaning_data)

In [50]:
data[['narrative', 'clean_narrative']].head()

,narrative,clean_narrative
0,purchase order day shipping amount receive pro...,purchase order day shipping amount receive pro...
1,forwarded message date tue subject please inve...,forwarded message date tue subject please inve...
2,forwarded message cc sent friday pdt subject f...,forwarded message sent friday pdt subject fina...
3,payment history missing credit report speciali...,payment history missing credit report speciali...
4,payment history missing credit report made mis...,payment history missing credit report made mis...


In [51]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['clean_narrative'])
tokenizer

In [55]:
sequences = tokenizer.texts_to_sequences(data['clean_narrative'])
# max_sequence_length = max(len(x) for x in sequences)
max_sequence_length = 250
x = pad_sequences(sequences, maxlen = max_sequence_length)
y = data['label'].values
x

array([[   0,    0,    0, ...,  304, 4387, 1414],
       [   0,    0,    0, ...,  103,  428,  295],
       [   0,    0,    0, ...,    5,  182,  163],
       ...,
       [   0,    0,    0, ...,   79,   13,  301],
       [   0,    0,    0, ..., 1197,   60,  945],
       [   0,    0,    0, ...,    9, 1509,  739]], dtype=int32)

In [52]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
vocab_size = len(tokenizer.word_index) + 1
vocab_size

45472

In [57]:
print(x_train.shape,y_train.shape)
print(x_test.shape,y_test.shape)

(129924, 250) (129924,)
(32481, 250) (32481,)


In [33]:
data['label'].value_counts()

label
1    91171
2    23145
3    18990
0    15564
4    13535
Name: count, dtype: int64

In [53]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

print(class_weights)

{np.int64(0): np.float64(2.089986326711172), np.int64(1): np.float64(0.35584892224261183), np.int64(2): np.float64(1.4101481521680144), np.int64(3): np.float64(1.711102331094429), np.int64(4): np.float64(2.3935887988209283)}


The dataset was imbalanced, with one class containing significantly more samples than the others. To reduce model bias toward the majority class, class weights were computed using compute_class_weight from scikit-learn and applied during training.

# RNN

In [54]:
embedding_dim = 100

rnn_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length= max_sequence_length),
    SimpleRNN(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(5,activation='softmax')
])

optimizer = Adam(learning_rate=0.0005)
rnn_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.0005,
    restore_best_weights=True
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1784736909.615368      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11352 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784736909.617757      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 12442 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [56]:
rnn_model.fit(x_train, 
            y_train, 
            epochs=30,
            batch_size=64,
            validation_data=(x_test,y_test), 
            callbacks=[early_stop],
            class_weight=class_weights)

Epoch 1/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 50s 24ms/step - accuracy: 0.2181 - loss: 1.6296 - val_accuracy: 0.1975 - val_loss: 1.6006
Epoch 2/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.2869 - loss: 1.5995 - val_accuracy: 0.2420 - val_loss: 1.6039
Epoch 3/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.3605 - loss: 1.5600 - val_accuracy: 0.4586 - val_loss: 1.3824
Epoch 4/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.4312 - loss: 1.4967 - val_accuracy: 0.4651 - val_loss: 1.3128
Epoch 5/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.4710 - loss: 1.4114 - val_accuracy: 0.5219 - val_loss: 1.1790
Epoch 6/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.5098 - loss: 1.3149 - val_accuracy: 0.4813 - val_loss: 1.2275
Epoch 7/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.5295 - loss: 1.2600 - val_accuracy: 0.5680 - val_loss: 1.0763
Epoch 8/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.5645 -

In [57]:
# حفظ الموديل
rnn_model.save("rnn_model.keras")

In [60]:
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

model = load_model("/kaggle/working/rnn_model.keras")
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# إذا y_test One-Hot
y_true = y_test
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted"))
print("Recall   :", recall_score(y_true, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_true, y_pred, average="weighted"))

print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

1016/1016 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step
Accuracy : 0.6902188972014408
Precision: 0.737290211273402
Recall   : 0.6902188972014408
F1 Score : 0.7029385324679202
                     precision    recall  f1-score   support

        credit_card       0.44      0.43      0.44      3131
   credit_reporting       0.92      0.73      0.81     18149
    debt_collection       0.48      0.69      0.57      4718
mortgages_and_loans       0.61      0.70      0.65      3804
     retail_banking       0.50      0.72      0.59      2679

           accuracy                           0.69     32481
          macro avg       0.59      0.65      0.61     32481
       weighted avg       0.74      0.69      0.70     32481



# LSTM

In [61]:
lstm_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_sequence_length),
    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(5, activation='softmax')
])

optimizer = Adam(learning_rate=0.0005)
lstm_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.0005,
    restore_best_weights=True
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [62]:
lstm_model.fit(x_train, y_train, 
              epochs=30, batch_size=64, 
              validation_data=(x_test,y_test), 
              callbacks=[early_stop],
              class_weight=class_weights)

Epoch 1/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1476s 724ms/step - accuracy: 0.7510 - loss: 0.7458 - val_accuracy: 0.8202 - val_loss: 0.5502
Epoch 2/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1471s 724ms/step - accuracy: 0.8128 - loss: 0.5462 - val_accuracy: 0.8305 - val_loss: 0.5192
Epoch 3/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1465s 721ms/step - accuracy: 0.8305 - loss: 0.4853 - val_accuracy: 0.8454 - val_loss: 0.4688
Epoch 4/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1450s 714ms/step - accuracy: 0.8439 - loss: 0.4400 - val_accuracy: 0.8417 - val_loss: 0.4832
Epoch 5/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1452s 715ms/step - accuracy: 0.8535 - loss: 0.4047 - val_accuracy: 0.8429 - val_loss: 0.4597
Epoch 6/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1451s 715ms/step - accuracy: 0.8628 - loss: 0.3716 - val_accuracy: 0.8490 - val_loss: 0.4383
Epoch 7/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1446s 712ms/step - accuracy: 0.8695 - loss: 0.3461 - val_accuracy: 0.8484 - val_loss: 0.4493
Epoch 8/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1449s 713ms/s

In [63]:
lstm_model.save("lstm_model.keras")

In [65]:
model = load_model("/kaggle/working/lstm_model.keras")
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)

y_true = y_test
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted"))
print("Recall   :", recall_score(y_true, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_true, y_pred, average="weighted"))

print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

1016/1016 ━━━━━━━━━━━━━━━━━━━━ 134s 132ms/step
Accuracy : 0.8679227856285213
Precision: 0.8766755544539615
Recall   : 0.8679227856285213
F1 Score : 0.8704790594397456
                     precision    recall  f1-score   support

        credit_card       0.69      0.87      0.77      3131
   credit_reporting       0.95      0.89      0.92     18149
    debt_collection       0.77      0.81      0.79      4718
mortgages_and_loans       0.81      0.86      0.83      3804
     retail_banking       0.87      0.86      0.86      2679

           accuracy                           0.87     32481
          macro avg       0.82      0.86      0.84     32481
       weighted avg       0.88      0.87      0.87     32481



# GRU

In [22]:
gru_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_sequence_length),
    GRU(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(5, activation='softmax')
])

optimizer = Adam(learning_rate=0.0005)
gru_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.0005,
    restore_best_weights=True
)

In [23]:
gru_model.fit(x_train, y_train, 
              epochs=30, batch_size=64, 
              validation_data=(x_test,y_test), 
              callbacks=[early_stop])

Epoch 1/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1365s 668ms/step - accuracy: 0.7704 - loss: 0.6252 - val_accuracy: 0.8628 - val_loss: 0.3970
Epoch 2/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1355s 667ms/step - accuracy: 0.8721 - loss: 0.3741 - val_accuracy: 0.8770 - val_loss: 0.3531
Epoch 3/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1363s 671ms/step - accuracy: 0.8859 - loss: 0.3279 - val_accuracy: 0.8799 - val_loss: 0.3429
Epoch 4/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1341s 660ms/step - accuracy: 0.8974 - loss: 0.2963 - val_accuracy: 0.8812 - val_loss: 0.3429
Epoch 5/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1358s 669ms/step - accuracy: 0.9059 - loss: 0.2718 - val_accuracy: 0.8824 - val_loss: 0.3395
Epoch 6/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1344s 662ms/step - accuracy: 0.9151 - loss: 0.2484 - val_accuracy: 0.8829 - val_loss: 0.3448
Epoch 7/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1338s 659ms/step - accuracy: 0.9215 - loss: 0.2288 - val_accuracy: 0.8824 - val_loss: 0.3509
Epoch 8/30
2031/2031 ━━━━━━━━━━━━━━━━━━━━ 1355s 667ms/s

In [24]:
gru_model.save("gru_model.keras")

In [33]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

model = load_model("/kaggle/input/models/miskbadr/gru-model/keras/default/1/gru_model.keras")
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# إذا y_test One-Hot
y_true = y_test
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted"))
print("Recall   :", recall_score(y_true, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_true, y_pred, average="weighted"))

print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

1016/1016 ━━━━━━━━━━━━━━━━━━━━ 45s 44ms/step
Accuracy : 0.8824235707028725
Precision: 0.8807604612047879
Recall   : 0.8824235707028725
F1 Score : 0.880985029969016
                     precision    recall  f1-score   support

        credit_card       0.81      0.76      0.79      3131
   credit_reporting       0.91      0.95      0.93     18149
    debt_collection       0.83      0.76      0.79      4718
mortgages_and_loans       0.88      0.82      0.85      3804
     retail_banking       0.86      0.88      0.87      2679

           accuracy                           0.88     32481
          macro avg       0.86      0.83      0.85     32481
       weighted avg       0.88      0.88      0.88     32481



# transformers

In [10]:
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [11]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased", device_map="auto")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
X_train_text, X_test_text, y_train_hf, y_test_hf = train_test_split(
    data["clean_narrative"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

In [15]:
train_dataset = Dataset.from_dict({
    "text": X_train_text.tolist(),
    "label": y_train_hf.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test_text.tolist(),
    "label": y_test_hf.tolist()
})

In [16]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/129924 [00:00<?, ? examples/s]

Map:   0%|          | 0/32481 [00:00<?, ? examples/s]

In [20]:
training_args = TrainingArguments(
    output_dir="./transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100
)

In [21]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.698034,0.724318,0.871556,0.871565,0.871556,0.867459
2,0.575014,0.629968,0.893014,0.891738,0.893014,0.891858
3,0.439433,0.633183,0.896986,0.896021,0.896986,0.896223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=12183, training_loss=0.6313251629062037, metrics={'train_runtime': 3106.2292, 'train_samples_per_second': 125.481, 'train_steps_per_second': 3.922, 'total_flos': 1.290871131591168e+16, 'train_loss': 0.6313251629062037, 'epoch': 3.0})

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/129924 [00:00<?, ? examples/s]

Map:   0%|          | 0/32481 [00:00<?, ? examples/s]

In [24]:
print(train_dataset)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 129924
})


In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100
)

In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [29]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [30]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.694648,0.706656,0.874665,0.874427,0.874665,0.871140
2,0.576036,0.630084,0.890890,0.889538,0.890890,0.889827
3,0.423191,0.637854,0.895077,0.894179,0.895077,0.894417


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=12183, training_loss=0.6305110646147432, metrics={'train_runtime': 3129.6221, 'train_samples_per_second': 124.543, 'train_steps_per_second': 3.893, 'total_flos': 1.290871131591168e+16, 'train_loss': 0.6305110646147432, 'epoch': 3.0})

In [31]:
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6300584673881531, 'eval_accuracy': 0.8908900588036083, 'eval_precision': 0.889539525619517, 'eval_recall': 0.8908900588036083, 'eval_f1': 0.8898275639984238, 'eval_runtime': 79.615, 'eval_samples_per_second': 407.976, 'eval_steps_per_second': 12.761, 'epoch': 3.0}


In [32]:
trainer.evaluate(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6300584673881531,
 'eval_accuracy': 0.8908900588036083,
 'eval_precision': 0.889539525619517,
 'eval_recall': 0.8908900588036083,
 'eval_f1': 0.8898275639984238,
 'eval_runtime': 80.4103,
 'eval_samples_per_second': 403.941,
 'eval_steps_per_second': 12.635,
 'epoch': 3.0}

In [43]:
predictions = trainer.predict(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [33]:
trainer.save_model("final_distilbert")

tokenizer.save_pretrained("final_distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_distilbert/tokenizer_config.json', 'final_distilbert/tokenizer.json')

In [35]:
import shutil

shutil.make_archive("final_distilbert", "zip", "final_distilbert")

'/kaggle/working/final_distilbert.zip'

In [36]:
from transformers import AutoTokenizer

hf_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
hf_tokenizer.save_pretrained("final_distilbert")

('final_distilbert/tokenizer_config.json', 'final_distilbert/tokenizer.json')

In [37]:
import shutil

shutil.make_archive(
    "/kaggle/working/final_distilbert",
    "zip",
    "/kaggle/working/final_distilbert"
)

'/kaggle/working/final_distilbert.zip'